# 64 — Precision & Recall for NLP
**Goal:** Measure entity extraction accuracy with precision, recall, and F1.

## 1. Why Metrics Matter

In [ ]:
print('''For resume NLP, we care about:
- Precision: Of the skills we found, how many were correct?
- Recall: Of the actual skills, how many did we find?
- F1: Harmonic mean of both

Example: Resume has [Python, Java, TensorFlow]
System finds: [Python, TensorFlow, Docker]
Precision = 2/3 = 0.67 (Docker was wrong)
Recall = 2/3 = 0.67 (missed Java)
F1 = 0.67''')

## 2. Computing Metrics

In [ ]:
def compute_metrics(gold, predicted):
    """Compute precision, recall, F1 for entity extraction."""
    gold_set = set(gold)
    pred_set = set(predicted)
    
    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}

# Test
test_cases = [
    ([], [], "empty"),
    (["Python", "Java"], ["Python", "Java"], "perfect"),
    (["Python", "Java", "SQL"], ["Python", "TensorFlow"], "partial"),
    (["Python"], [], "no prediction"),
    ([], ["Python"], "hallucination"),
]
for gold, pred, name in test_cases:
    m = compute_metrics(gold, pred)
    print(f"""{name:15s} P={m['precision']:.2f} R={m['recall']:.2f} F1={m['f1']:.2f} (tp={m['tp']} fp={m['fp']} fn={m['fn']})""")

## 3. Per-Category Breakdown

In [ ]:
# Evaluate skill extraction per category
categories = {
    "Python": "programming",
    "Java": "programming",
    "NLP": "nlp",
    "TensorFlow": "ml",
    "AWS": "cloud",
}
gold = ["Python", "Java", "NLP", "TensorFlow", "AWS"]
predicted = ["Python", "NLP", "TensorFlow", "Docker", "SQL"]  # Docker and SQL are wrong

from collections import defaultdict
cat_metrics = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

for item in gold:
    cat = categories.get(item, "other")
    if item in predicted:
        cat_metrics[cat]["tp"] += 1
    else:
        cat_metrics[cat]["fn"] += 1

for item in predicted:
    cat = categories.get(item, "other")
    if item not in gold:
        cat_metrics[cat]["fp"] += 1

print("Per-category breakdown:")
for cat, m in sorted(cat_metrics.items()):
    p = m["tp"] / (m["tp"] + m["fp"]) if (m["tp"] + m["fp"]) > 0 else 0
    r = m["tp"] / (m["tp"] + m["fn"]) if (m["tp"] + m["fn"]) > 0 else 0
    f1 = 2*p*r/(p+r) if (p+r) > 0 else 0
    print(f"  {cat:12s} P={p:.2f} R={r:.2f} F1={f1:.2f}")

## Summary: Precision, recall, F1 are the standard NLP evaluation metrics. Track per-category for insights.